# Step 4 — BAGEL CFG branch-pruning gate

This notebook tests where BAGEL's refinement compute can be removed before combining that optimization with continuous batching. One loaded M* server runs five policies on identical prompt/seed inputs: the unchanged three-branch reference, image-branch pruning, late-step pruning, their automatic combination, and an intentionally unguided main-only negative control.

It saves post-step latents after refinement steps 0, 20, 40, 41, and 48; compares final images; records per-step CUDA time; and draws a four-panel diagnostic plot. Compilation is explicitly disabled for this recovery run, and latent capture synchronizes selected steps, so this is a correctness and branch-work gate—not performance evidence.

## 1. Mount Drive and define the run

Open this notebook in Cursor, choose the **Colab** kernel, and connect an **A100 High-RAM** runtime. The code still executes on Colab; Cursor is only the interface.

In [ ]:
from google.colab import drive
from datetime import datetime, timezone
from pathlib import Path
import json, os, shutil, subprocess, sys

drive.mount('/content/drive')
RUN_ID = 'cfg_branch_gate_' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
FORK_URL = 'https://github.com/yegurla-sumanth/mstar.git'
RUN_COMMIT = '751b483336adaac412446ad4ca4c36c701b4991d'
REPO = Path('/content') / f'mstar_{RUN_ID}'
CACHE_DIR = Path('/content/hf_cache_bagel')
RESULT_DIR = Path('/content/drive/MyDrive/CMU_11868_BAGEL') / RUN_ID
GATE_DIR = RESULT_DIR / 'cfg_branch_gate_b1_eager_correctness'
RESULT_DIR.mkdir(parents=True, exist_ok=False)
print({'run_id': RUN_ID, 'repo': str(REPO), 'results': str(RESULT_DIR)})

## 2. Clone the immutable implementation and check the runtime

The experiment is pinned to the implementation commit, not the moving feature branch. The hardware preflight stops before large downloads if the runtime is unsuitable.

In [ ]:
subprocess.run(['git', 'clone', '--no-checkout', FORK_URL, str(REPO)], check=True)
subprocess.run(['git', 'checkout', '--detach', RUN_COMMIT], cwd=REPO, check=True)
head = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPO, text=True).strip()
assert head == RUN_COMMIT, (head, RUN_COMMIT)
source = {
    'fork_url': FORK_URL,
    'requested_commit': RUN_COMMIT,
    'resolved_commit': head,
    'git_status': subprocess.check_output(['git', 'status', '--porcelain=v1'], cwd=REPO, text=True),
}
(RESULT_DIR / 'source.json').write_text(json.dumps(source, indent=2) + '\n')
subprocess.run([
    sys.executable, str(REPO / 'experiments/bagel/scripts/environment_probe.py'),
    '--output', str(RESULT_DIR / 'preinstall_environment.json'),
    '--disk-path', '/content',
    '--require-gpu-memory-gib', '75',
    '--require-host-memory-gib', '50',
    '--require-disk-free-gib', '70',
], check=True)
print('Hardware preflight passed at commit', head)

## 3. Install and validate the pinned environment

This retains the working CUDA 12.8, FlashInfer, Ninja, and noninteractive Matplotlib setup from Steps 1–3.

In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', 'uv==0.10.0'], check=True)
UV = subprocess.check_output([sys.executable, '-c', 'import shutil; print(shutil.which("uv"))'], text=True).strip()
VENV = REPO / '.venv-bagel'
subprocess.run([UV, 'venv', '--python', '3.12', '--seed', str(VENV)], check=True)
VENV_PY = VENV / 'bin/python'
constraints = REPO / 'experiments/bagel/constraints-colab-cu128.txt'
subprocess.run([
    UV, 'pip', 'install', '--python', str(VENV_PY), '--torch-backend=cu128',
    '--constraint', str(constraints), '-e', f'{REPO}[bagel]', 'matplotlib==3.10.7',
], check=True)
subprocess.run([UV, 'pip', 'check', '--python', str(VENV_PY)], check=True)
ninja_path = VENV / 'bin/ninja'
assert ninja_path.exists(), f'Missing required FlashInfer JIT tool: {ninja_path}'
subprocess.run([str(ninja_path), '--version'], check=True)
for test_path in (
    'test/modular/test_bagel_checkpoint_revision.py',
    'test/modular/test_execution_trace.py',
    'test/modular/test_bagel_image_batch_packing.py',
    'test/modular/test_bagel_batch_parity.py',
):
    subprocess.run([str(VENV_PY), str(REPO / test_path), '-v'], cwd=REPO, check=True)
import_check = subprocess.check_output([
    str(VENV_PY), '-c',
    'import os; os.environ["MPLBACKEND"]="Agg"; import torch, flashinfer, matplotlib; assert torch.cuda.is_available(); x=torch.ones((16,16),device="cuda",dtype=torch.bfloat16); y=x@x; torch.cuda.synchronize(); print(torch.__version__, torch.version.cuda, flashinfer.__version__, matplotlib.__version__, float(y[0,0]))'
], text=True)
(RESULT_DIR / 'gpu_import_check.txt').write_text(import_check)
freeze = subprocess.check_output([UV, 'pip', 'freeze', '--python', str(VENV_PY)], text=True)
(RESULT_DIR / 'resolved_requirements.txt').write_text(freeze)
subprocess.run([
    str(VENV_PY), str(REPO / 'experiments/bagel/scripts/environment_probe.py'),
    '--output', str(RESULT_DIR / 'installed_environment.json'),
    '--disk-path', '/content',
    '--require-gpu-memory-gib', '75',
    '--require-host-memory-gib', '50',
    '--require-disk-free-gib', '60',
], check=True)
print('Environment, CPU tests, and GPU import preflight passed')

## 4. Reuse or download the pinned BAGEL checkpoint

In [ ]:
subprocess.run([
    str(VENV_PY), str(REPO / 'experiments/bagel/scripts/download_checkpoint.py'),
    '--cache-dir', str(CACHE_DIR),
    '--manifest', str(RESULT_DIR / 'checkpoint_manifest.json'),
], check=True)
import yaml
manifest = json.loads((RESULT_DIR / 'checkpoint_manifest.json').read_text())
pinned_config = yaml.safe_load((REPO / 'experiments/bagel/configs/bagel_single_gpu_pinned.yaml').read_text())
assert manifest['resolved_revision'] == pinned_config['model_kwargs']['checkpoint_revision']
print('Pinned checkpoint is available at', manifest['resolved_revision'])

## 5. Run the five-arm eager correctness gate

The orchestrator loads BAGEL once, performs one unmeasured reference warm-up, then runs each policy once at B=1. `TORCH_COMPILE_DISABLE=1` is intentional: this run establishes policy correctness after the compiled dynamic-branch failure and is not performance evidence. The engine log must say that compilation is globally disabled. The orchestrator always stops the M* process group in `finally`; do not rerun this cell while it is active.

In [ ]:
assert not GATE_DIR.exists(), f'Result directory already exists; create a fresh RUN_ID: {GATE_DIR}'
import socket
with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as probe_socket:
    try:
        probe_socket.bind(('127.0.0.1', 8024))
    except OSError as exc:
        raise RuntimeError('Port 8024 is busy. Stop the earlier M* process before this run.') from exc
gate_env = os.environ.copy()
gate_env['TORCH_COMPILE_DISABLE'] = '1'
subprocess.run([
    str(VENV_PY), str(REPO / 'experiments/bagel/scripts/run_cfg_branch_sweep.py'),
    '--repo-root', str(REPO),
    '--cache-dir', str(CACHE_DIR),
    '--output-dir', str(GATE_DIR),
    '--port', '8024',
    '--repeats', '1',
], cwd=REPO, env=gate_env, check=True)
print('Eager CFG correctness gate completed:', GATE_DIR)

## 6. Inspect numerical drift and branch work

The dashed boundary in the diagnostic plot separates step 40 (inside CFG) from step 41 (outside CFG). `main_only` is a negative control and is expected to diverge. We will interpret the measured errors before accepting `drop_image`, `late_main`, or `auto`; this notebook deliberately does not invent a tolerance after seeing the result. Timing from this eager, latent-capturing run is diagnostic only and must not be reported as a speedup.

In [ ]:
orchestrator = json.loads((GATE_DIR / 'cfg_orchestrator.json').read_text())
analysis = json.loads((GATE_DIR / 'cfg_branch_analysis.json').read_text())
assert orchestrator['status'] == 'success', orchestrator
assert analysis['status'] == 'complete', analysis
rows = []
for policy in analysis['policies']:
    item = analysis['policy_summary'][policy]
    comparison = analysis['comparisons'].get(policy, {})
    rows.append({
        'policy': policy,
        'expected_virtual_rows': item['expected_virtual_rows'],
        'eager_diagnostic_client_seconds': round(item['median_client_seconds'], 3),
        'image_psnr_db': comparison.get('minimum_image_psnr_db'),
        'max_latent_relative_l2': comparison.get('maximum_latent_relative_l2_error'),
    })
print(json.dumps({
    'resolved_commit': head,
    'server_ready_seconds': orchestrator['server_ready_seconds'],
    'policies': rows,
    'measurement_boundary': analysis['measurement_boundary'],
    'result_dir': str(GATE_DIR),
}, indent=2))

from IPython.display import Image, display
display(Image(filename=str(GATE_DIR / 'cfg_branch_analysis.png'), width=1000))
sweep = json.loads((GATE_DIR / 'cfg_sweep_summary.json').read_text())
for record in sweep['measured_requests']:
    print(record['cfg_branch_policy'], record['image']['sha256'])
    display(Image(filename=record['image']['path'], width=300))

## 7. Package the evidence

Download the ZIP before deleting the runtime. It contains raw traces, five images, selected latent tensors, logs, the resolved environment, JSON analysis, and the plot. Treat all timing as diagnostic because compilation is disabled and latent capture synchronizes selected steps.

In [ ]:
archive_base = RESULT_DIR.parent / f'{RUN_ID}_results'
archive_path = shutil.make_archive(
    str(archive_base), 'zip', root_dir=RESULT_DIR.parent, base_dir=RESULT_DIR.name
)
print('Saved ZIP:', archive_path)
from google.colab import files
files.download(archive_path)

After the ZIP download finishes and the Drive directory is visible, use **Runtime → Disconnect and delete runtime**. Share the printed summary and downloaded ZIP. Do not launch the repeated performance suite yet: selected-step latent capture intentionally perturbs timing, and we first need to decide which optimized policies pass the numerical gate.